In [ ]:
"""
figures_fcms.ipynb
===============
Reproduces Figures 2, 3, 4, and 5 from:

    Grassi, S. (2026). Feedback-Coupled Memory Systems: Dynamical Model
    for Adaptive Coordination.

Figure 2 — Disagreement dynamics and Neimark–Sacker bifurcation boundary
    Left panel:  Time series of disagreement d_t at three coupling values
                 (beta = 0.50, 1.41, 1.55), illustrating stable convergence,
                 critical slowing down, and oscillatory divergence.
    Right panel: Dominant eigenvalue magnitude lambda(beta) as a function of
                 coupling strength, showing monotonic approach to the unit
                 circle at the bifurcation threshold beta_c.

Figure 3 — Early warning signatures of coordination breakdown
    Left panel:  Variance of d_t under bounded noise as a function of beta,
                 showing a sharp increase as beta approaches beta_c.
    Right panel: Lag-1 autocorrelation of d_t approaching unity near beta_c.
    Both panels confirm the theoretical early warning predictions in Section 4.

Figure 4 — Phase portrait of the nonlinear FCMS under tanh saturation
    Multiple trajectories from different initial conditions converge to the
    coordination manifold d = 0, confirming that the dissipative-feedback
    mechanism remains effective under bounded environmental memory capacity.

Parameters used throughout (matching main text):
    gamma = 1.0   (memory dissipation rate)
    eta   = 0.1   (agent responsiveness)
    beta_c = sqrt(gamma / (4 * eta)) ≈ 1.581  (critical coupling threshold)

Dependencies: numpy, matplotlib
Python: 3.9+

Usage:
    python figures_fcms.ipynb

Figures are saved to the same directory as this script.

Repository: https://github.com/stevefatz95/dynamic-adaptive-coordination
"""

import os
import numpy as np

# ─── Detect environment before importing matplotlib ───────────────────────────
try:
    get_ipython  # noqa: F821 — defined in Jupyter kernels
    IN_NOTEBOOK = True
except NameError:
    IN_NOTEBOOK = False

import matplotlib
if not IN_NOTEBOOK:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ─── Output directory ─────────────────────────────────────────────────────────
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__)) if not IN_NOTEBOOK else os.getcwd()

# ─── Global style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.linewidth': 0.8,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
    'lines.linewidth': 1.4,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# ─── Core parameters ──────────────────────────────────────────────────────────
gamma  = 1.0
eta    = 0.1
T      = 120
beta_c = np.sqrt(gamma / (4 * eta))   # ≈ 1.581

# ─── Helper functions ─────────────────────────────────────────────────────────

def jacobian(beta, gamma, eta):
    return np.array([[1 - gamma, beta],
                     [-4 * eta * beta, 1]])

def spectral_radius(beta, gamma, eta):
    return max(abs(np.linalg.eigvals(jacobian(beta, gamma, eta))))

def simulate(beta, gamma, eta, T, S0=0.5, d0=2.0):
    S, d = S0, d0
    Ss, ds = [S], [d]
    for _ in range(T):
        S_new = (1 - gamma) * S + beta * d
        d_new = d - 4 * eta * beta * S
        S, d = S_new, d_new
        Ss.append(S)
        ds.append(d)
    return np.array(Ss), np.array(ds)

def simulate_nonlinear(beta, gamma, eta, T, S0, d0):
    S, d = S0, d0
    Ss, ds = [S], [d]
    for _ in range(T):
        S_new = (1 - gamma) * np.tanh(S) + beta * d
        d_new = d - 4 * eta * beta * S
        S, d = S_new, d_new
        Ss.append(S)
        ds.append(d)
    return np.array(Ss), np.array(ds)

def save_fig(fig, filename):
    path = os.path.join(SCRIPT_DIR, filename)
    fig.savefig(path)
    print(f"Saved: {path}")

# ─── FIGURE 2 — Disagreement dynamics and bifurcation boundary ───────────────
print("Generating Figure 2...")

fig2, axes = plt.subplots(1, 2, figsize=(10, 4.2))

betas_plot = [0.50, 1.41, 1.55]
styles     = ['-', '--', ':']
colors     = ['#1a6faf', '#e07b39', '#c0392b']
labels     = [r'$\beta = 0.50$  (stable)',
              r'$\beta = 1.41$  (near $\beta_c$)',
              r'$\beta = 1.55$  (unstable)']

ax0 = axes[0]
for b, ls, col, lbl in zip(betas_plot, styles, colors, labels):
    _, ds = simulate(b, gamma, eta, T)
    ds_plot = np.clip(ds, -25, 25)
    ax0.plot(np.arange(len(ds)), ds_plot, ls=ls, color=col, label=lbl, lw=1.5)

ax0.axhline(0, color='k', lw=0.6)
ax0.set_xlabel(r'Time $t$')
ax0.set_ylabel(r'Disagreement $d_t$')
ax0.set_title('Disagreement dynamics and critical slowing down')
ax0.legend(loc='upper right', frameon=False, fontsize=9)
ax0.set_xlim(0, T)
ax0.set_ylim(-15, 15)
ax0.text(T * 0.98, 1.2, r'$\beta_c \approx 1.58$', ha='right',
         fontsize=9, color='#555', style='italic')

ax1 = axes[1]
betas_range = np.linspace(0.0, 2.0, 400)
rhos = [spectral_radius(b, gamma, eta) for b in betas_range]
ax1.plot(betas_range, rhos, color='#1a6faf', lw=1.6, label=r'$\lambda(\beta)$')
ax1.axhline(1.0, color='k', lw=0.8, ls='--', label='Unit circle boundary')
ax1.axvline(beta_c, color='#c0392b', lw=0.9, ls=':', alpha=0.8,
            label=r'$\beta_c \approx 1.58$')
for b, col in zip(betas_plot, colors):
    ax1.scatter([b], [spectral_radius(b, gamma, eta)], color=col, s=45, zorder=5)
ax1.set_xlabel(r'Coupling strength $\beta$')
ax1.set_ylabel(r'Dominant eigenvalue magnitude $\lambda(\beta)$')
ax1.set_title('Neimark–Sacker bifurcation boundary')
ax1.legend(loc='upper left', frameon=False, fontsize=9)
ax1.set_xlim(0, 2.0)
ax1.set_ylim(0, 1.5)

fig2.tight_layout(pad=1.5)
save_fig(fig2, 'Figure_2.png')

# ─── FIGURE 3 — Early warning signatures ─────────────────────────────────────
print("Generating Figure 3...")

plt.rcParams.update({
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
})

N_noise   = 2000
noise_std = 0.05
betas_ews = np.linspace(0.3, 1.55, 40)
variances, autocorrs = [], []

rng = np.random.default_rng(42)
for b in betas_ews:
    d_series = []
    S, d = 0.0, 0.1
    for _ in range(N_noise):
        noise = rng.normal(0, noise_std)
        S_new = (1 - gamma) * S + b * d
        d_new = d - 4 * eta * b * S + noise
        S, d = S_new, d_new
        d_series.append(d)
    d_arr = np.array(d_series[500:])
    variances.append(np.var(d_arr))
    autocorrs.append(np.corrcoef(d_arr[:-1], d_arr[1:])[0, 1] if len(d_arr) > 1 else np.nan)

variances = np.array(variances)
autocorrs = np.array(autocorrs)

fig3, (ax_v, ax_a) = plt.subplots(1, 2, figsize=(13, 4.8))

ax_v.plot(betas_ews, variances, color='#1a6faf', lw=1.6)
ax_v.axvline(beta_c, color='#c0392b', lw=0.9, ls=':', alpha=0.9,
             label=r'$\beta_c \approx 1.58$')
ax_v.set_xlabel(r'Coupling strength $\beta$')
ax_v.set_ylabel(r'Variance of $d_t$')
ax_v.set_title(r'Rising variance near $\beta_c$')
ax_v.legend(frameon=False)

ax_a.plot(betas_ews, autocorrs, color='#e07b39', lw=1.6)
ax_a.axhline(1.0, color='k', lw=0.6, ls='--', alpha=0.5, label='AC = 1')
ax_a.axvline(beta_c, color='#c0392b', lw=0.9, ls=':', alpha=0.9,
             label=r'$\beta_c \approx 1.58$')
ax_a.set_xlabel(r'Coupling strength $\beta$')
ax_a.set_ylabel(r'Lag-1 autocorrelation of $d_t$')
ax_a.set_title(r'Increasing autocorrelation near $\beta_c$')
ax_a.legend(frameon=False)

fig3.tight_layout(pad=1.5)
save_fig(fig3, 'Figure_3.png')

# ─── FIGURE 4 — Nonlinear phase portrait ─────────────────────────────────────
print("Generating Figure 4...")

fig4, ax = plt.subplots(figsize=(6.0, 5.2))
beta_nl = 1.2
T_nl    = 200
ICs     = [(0.8, 3.0), (-0.8, -3.0), (1.5, 1.0), (-1.5, -1.0),
           (0.3, -2.5), (-0.3, 2.5)]
cmap    = plt.cm.Blues

for k, (S0, d0) in enumerate(ICs):
    Ss, ds = simulate_nonlinear(beta_nl, gamma, eta, T_nl, S0, d0)
    col = cmap(0.45 + 0.45 * k / len(ICs))
    ax.plot(Ss, ds, color=col, lw=1.0, alpha=0.85)
    mid = len(Ss) // 3
    ax.annotate('', xy=(Ss[mid + 1], ds[mid + 1]), xytext=(Ss[mid], ds[mid]),
                arrowprops=dict(arrowstyle='->', color=col, lw=0.9))

S_grid = np.linspace(-2.0, 2.0, 18)
d_grid = np.linspace(-4.5, 4.5, 18)
SG, DG = np.meshgrid(S_grid, d_grid)
dS = (1 - gamma) * np.tanh(SG) + beta_nl * DG - SG
dD = -4 * eta * beta_nl * SG
norm = np.sqrt(dS**2 + dD**2) + 1e-10
ax.quiver(SG, DG, dS / norm, dD / norm,
          alpha=0.22, color='#555', scale=28, width=0.003)

ax.scatter([0], [0], color='#c0392b', s=60, zorder=6, label='Fixed point $(0,0)$')
ax.axvline(0, color='k', lw=0.5, ls='--', alpha=0.4)
ax.axhline(0, color='k', lw=0.5, ls='--', alpha=0.4,
           label=r'Coord. manifold $d=0$')
ax.set_xlabel(r'Environmental state $S_t$')
ax.set_ylabel(r'Disagreement $d_t$')
ax.set_title(r'Phase portrait: nonlinear FCMS ($\tanh$ saturation, $\beta = 1.2$)')
ax.legend(loc='upper right', frameon=False, fontsize=9.5)
ax.set_xlim(-2.1, 2.1)
ax.set_ylim(-5.0, 5.0)

fig4.tight_layout(pad=1.0)
save_fig(fig4, 'Figure_4.png')

print("\nAll figures generated successfully.")

# ─── FIGURE 5 — Scalability: synchronization variance vs population size ──────
print("Generating Figure 5...")

def simulate_meanfield(N, beta, gamma, eta, T_run=300, seed=42):
    """Mean-field FCMS: N agents share a single environmental state S.
    Each agent receives incentive G = -2*beta*S and updates x += eta*G.
    Environmental state tracks aggregate spread: S = (1-gamma)*S + beta*std(x).
    Returns final variance of agent states as synchronization measure.
    """
    rng_mf = np.random.default_rng(seed)
    x = rng_mf.normal(0, 1, N)
    S = 0.0
    for _ in range(T_run):
        G = -2 * beta * S * np.ones(N)
        x = x + eta * G
        S = (1 - gamma) * S + beta * np.std(x)
    return np.var(x)

N_values  = np.logspace(1, 6, 20).astype(int)
beta_mf   = 0.5   # stable regime: 4*eta*beta^2 = 0.1 < gamma = 1.0
sync_vars = [simulate_meanfield(N, beta=beta_mf, gamma=gamma, eta=eta)
             for N in N_values]

ref_curve = sync_vars[0] * N_values[0] / N_values

fig5, ax5 = plt.subplots(figsize=(6.5, 4.8))

ax5.loglog(N_values, sync_vars, 'o-', color='#1a6faf', lw=1.6,
           ms=5, label='Synchronization variance')
ax5.loglog(N_values, ref_curve, '--', color='#aaa', lw=1.2,
           label=r'$\sim 1/N$ reference')
ax5.axhline(1.3e-7, color='#c0392b', lw=0.9, ls=':',
            label=r'$N=10^6$ baseline ($\approx 1.3\times10^{-7}$)')

ax5.set_xlabel(r'Population size $N$')
ax5.set_ylabel(r'Synchronization variance')
ax5.set_title(r'Dimension-invariance of the dissipative-feedback criterion')
ax5.legend(frameon=False, fontsize=9.5)

fig5.tight_layout(pad=1.2)
save_fig(fig5, 'Figure_5.png')

print("Figure 5 saved")
print("\nAll figures generated successfully.")